# Day 12：HTTP 请求与大模型 API 客户端（Key Day!）

> **项目里程碑**：`v0.1.0-day12` | **核心技术**：`httpx`、`Authorization: Bearer`、JSON 序列化、HTTP 状态码异常处理、`httpx.MockTransport` 离线测试

本 Notebook 全程使用 `httpx.MockTransport` 在本地内存模拟 OpenAI/DeepSeek `/v1/chat/completions` 接口，**无需外网与真实 API Key** 即可从头到尾完整运行。

## 1. 第一步：构造标准对话消息 `ChatMessage` 与异常体系

In [1]:
import json
import httpx

VALID_ROLES = ("system", "user", "assistant", "tool")

class ChatMessage:
    """标准对话消息对象，支持转为 OpenAI/DeepSeek API 所需的字典格式。"""
    def __init__(self, role: str, content: str) -> None:
        if role not in VALID_ROLES:
            raise ValueError(f"无效角色: {role}")
        if not content.strip():
            raise ValueError("content 不能为空")
        self.role = role
        self.content = content

    def to_dict(self) -> dict[str, str]:
        return {"role": self.role, "content": self.content}

    def __str__(self) -> str:
        return f"[{self.role}] {self.content}"

class LLMAPIError(Exception):
    def __init__(self, message: str, status_code: int | None = None) -> None:
        self.status_code = status_code
        super().__init__(f"[HTTP {status_code}] {message}" if status_code else message)

class AuthenticationError(LLMAPIError): pass
class RateLimitError(LLMAPIError): pass
class ServerError(LLMAPIError): pass

print("消息模型与异常体系初始化完成！")

消息模型与异常体系初始化完成！


## 2. 第二步：构造请求头（Headers）与请求体（Payload）

In [2]:
def build_headers(api_key: str) -> dict[str, str]:
    return {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

def build_payload(messages: list[ChatMessage], model: str = "deepseek-chat", temperature: float = 0.7) -> dict:
    return {
        "model": model,
        "messages": [m.to_dict() for m in messages],
        "temperature": temperature,
        "stream": False,
    }

sample_msgs = [
    ChatMessage("system", "你是星辰数科智枢平台架构师。"),
    ChatMessage("user", "请用一句话解释 HTTP POST 请求的作用。"),
]
print("Headers:", build_headers("sk-demo-***"))
print("Payload:", json.dumps(build_payload(sample_msgs, temperature=0.3), ensure_ascii=False, indent=2))

Headers: {'Authorization': 'Bearer sk-demo-***', 'Content-Type': 'application/json'}
Payload: {
  "model": "deepseek-chat",
  "messages": [
    {
      "role": "system",
      "content": "你是星辰数科智枢平台架构师。"
    },
    {
      "role": "user",
      "content": "请用一句话解释 HTTP POST 请求的作用。"
    }
  ],
  "temperature": 0.3,
  "stream": false
}


## 3. 第三步：封装 `LLMClient` 并注入 `httpx.MockTransport`

In [3]:
class LLMClient:
    def __init__(self, api_key: str, base_url: str = "https://api.deepseek.com", model: str = "deepseek-chat", transport: httpx.BaseTransport | None = None) -> None:
        self.api_key = api_key
        self.model = model
        self._client = httpx.Client(base_url=base_url.rstrip("/"), timeout=30.0, transport=transport, trust_env=False)

    def chat(self, messages: list[ChatMessage], temperature: float = 0.7) -> ChatMessage:
        resp = self._client.post(
            "/v1/chat/completions",
            headers=build_headers(self.api_key),
            json=build_payload(messages, model=self.model, temperature=temperature),
        )
        if resp.status_code == 401:
            raise AuthenticationError("API Key 无效或已过期", status_code=401)
        if resp.status_code == 429:
            raise RateLimitError("触发限流，请稍后重试", status_code=429)
        if 500 <= resp.status_code < 600:
            raise ServerError("大模型服务端内部错误", status_code=resp.status_code)
        if resp.status_code >= 400:
            raise LLMAPIError(resp.text, status_code=resp.status_code)

        data = resp.json()
        content = data["choices"][0]["message"]["content"]
        return ChatMessage(role="assistant", content=content)

def mock_gateway(request: httpx.Request) -> httpx.Response:
    auth = request.headers.get("Authorization", "")
    if auth == "Bearer sk-bad":
        return httpx.Response(401, json={"error": {"message": "Invalid API key"}})
    if auth == "Bearer sk-limit":
        return httpx.Response(429, json={"error": {"message": "Rate limit exceeded"}})
    if auth == "Bearer sk-500":
        return httpx.Response(500, json={"error": {"message": "Internal Server Error"}})
    body = json.loads(request.content.decode("utf-8"))
    question = body["messages"][-1]["content"]
    return httpx.Response(200, json={
        "choices": [{"message": {"role": "assistant", "content": f"[Mock:{body['model']}] HTTP POST 用于向服务器提交数据：{question}"}}]
    })

transport = httpx.MockTransport(mock_gateway)
client = LLMClient(api_key="sk-valid", transport=transport)
reply = client.chat(sample_msgs, temperature=0.3)
print("正常回复:", reply)

正常回复: [assistant] [Mock:deepseek-chat] HTTP POST 用于向服务器提交数据：请用一句话解释 HTTP POST 请求的作用。


## 4. 第四步：验证 `401`、`429`、`500` 状态码异常精准捕获

In [4]:
for test_key, label in [("sk-bad", "401 鉴权失败"), ("sk-limit", "429 触发限流"), ("sk-500", "500 服务端错误")]:
    err_client = LLMClient(api_key=test_key, transport=transport)
    try:
        err_client.chat([ChatMessage("user", "ping")])
    except LLMAPIError as exc:
        print(f"[{label}] 捕获 {type(exc).__name__}: {exc}")

[401 鉴权失败] 捕获 AuthenticationError: [HTTP 401] API Key 无效或已过期
[429 触发限流] 捕获 RateLimitError: [HTTP 429] 触发限流，请稍后重试
[500 服务端错误] 捕获 ServerError: [HTTP 500] 大模型服务端内部错误
